![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Week 2, Day 3 -- Lab 3: Image Search with a Frozen EfficientNet

A pretrained model can turn any image into a short list of numbers -- an
**embedding** -- that captures what's in it. Similar images get similar numbers.

We **freeze** EfficientNet (use it without training -- pure transfer learning),
turn a small set of photos into embeddings, and find look-alikes with **cosine
similarity**.

## 1) Setup

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
import urllib.request, zipfile
from torchvision import datasets
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

## 2) Load a frozen EfficientNet (no training)

We drop its classifier so the model outputs the **embedding** instead of a label.

In [ ]:
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)
model.classifier = torch.nn.Identity()   # keep the features, remove the classifier
model.eval();

## 3) Get a small set of photos (~45 MB)

A few hundred real photos of ants and bees.

In [ ]:
urllib.request.urlretrieve("https://download.pytorch.org/tutorial/hymenoptera_data.zip", "data.zip")
zipfile.ZipFile("data.zip").extractall()

In [ ]:
prep = weights.transforms()
gallery = datasets.ImageFolder("hymenoptera_data/train", transform=prep)
shown   = datasets.ImageFolder("hymenoptera_data/train")   # same images, for display
print(len(gallery), "images")

## 4) Turn every image into an embedding

In [ ]:
vectors = []
with torch.no_grad():
    for i in range(0, len(gallery), 50):
        batch = torch.stack([gallery[j][0] for j in range(i, min(i + 50, len(gallery)))])
        vectors.append(model(batch))
embeddings = torch.cat(vectors).numpy()
print("embeddings:", embeddings.shape)

## 5) Simplify with PCA

PCA squeezes each 1280-number embedding down to 50 -- faster and less noisy.

In [ ]:
reduced = PCA(n_components=50).fit_transform(embeddings)

## 6) Search for the 5 most similar images

In [ ]:
query = 0                                       # try other numbers!
scores = cosine_similarity([reduced[query]], reduced)[0]
top5 = scores.argsort()[::-1][1:6]              # most similar (skip the query itself)

## 7) Show the query and its look-alikes

In [ ]:
plt.figure(figsize=(12, 3))
plt.subplot(1, 6, 1); plt.imshow(shown[query][0]); plt.title("query"); plt.axis("off")
for j, idx in enumerate(top5):
    plt.subplot(1, 6, j + 2); plt.imshow(shown[idx][0]); plt.title(f"#{j+1}"); plt.axis("off")
plt.show()

## Try it yourself

- Change `query` to a different number and re-run steps 6-7.
- The model was never trained on this task -- it just *understands* images well
  enough that similar ones land near each other. That's transfer learning.

### *Contributed By: Sattam Altwaim*